In [1]:
import os
import numpy as np
import pandas as pd

POSSIBLE_PATHS = [
    os.path.join(os.getcwd(), "data", "osic-pulmonary-fibrosis-progression"),
    os.path.join("/", "kaggle", "input", "osic-pulmonary-fibrosis-progression"),
    "./data/osic-pulmonary-fibrosis-progression",
    "./input/osic-pulmonary-fibrosis-progression",
]

BASE_PATH = None
for p in POSSIBLE_PATHS:
    if os.path.isdir(p):
        BASE_PATH = p
        break
if BASE_PATH is None:
    raise FileNotFoundError("Could not locate the dataset directory.")


def load_csv(fname):
    path = os.path.join(BASE_PATH, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")
    return pd.read_csv(path)


TRAIN_DF = load_csv("train.csv")
TEST_DF = load_csv("test.csv")
SAMPLE_SUBMISSION = load_csv("sample_submission.csv")

patient_trend = {}
global_slopes = []
for patient_id, grp in TRAIN_DF.groupby("Patient"):
    if len(grp) >= 2:
        slope, intercept = np.polyfit(grp["Weeks"].values, grp["FVC"].values, 1)
        slope = np.clip(slope, -200.0, 200.0)
        patient_trend[patient_id] = (intercept, slope)
        global_slopes.append(slope)
    else:
        intercept = grp["FVC"].values[0]
        patient_trend[patient_id] = (intercept, 0.0)

avg_slope = np.clip(np.mean(global_slopes) if global_slopes else 0.0, -200.0, 200.0)


def _predict_fvc(patient, week):
    """Base prediction used for bias estimation and test inference."""
    if patient in patient_trend:
        intercept, slope = patient_trend[patient]
    else:
        intercept = TRAIN_DF["FVC"].mean()
        slope = avg_slope
    slope = np.clip(slope, -200.0, 200.0)
    pred = intercept + slope * week
    pred = np.clip(pred, 827.0, 6399.0)  # FVC bounds from MIN_MAX
    pred = max(pred, 0.0)
    return pred


train_residuals = []
for _, row in TRAIN_DF.iterrows():
    pred = _predict_fvc(row["Patient"], row["Weeks"])
    train_residuals.append(row["FVC"] - pred)

BIAS = float(np.median(train_residuals))

MEDIAN_ABS_ERROR = np.median(np.abs(np.array(train_residuals) - BIAS))

CONSTANT_CONFIDENCE = max(70.0, MEDIAN_ABS_ERROR)

patient_bias = {}
patient_confidence = {}
for patient_id, grp in TRAIN_DF.groupby("Patient"):
    residuals = []
    for _, row in grp.iterrows():
        pred = _predict_fvc(row["Patient"], row["Weeks"])
        residuals.append(row["FVC"] - pred)
    pat_bias = float(np.median(residuals))
    pat_conf = max(70.0, np.median(np.abs(np.array(residuals) - pat_bias)))
    patient_bias[patient_id] = pat_bias
    patient_confidence[patient_id] = pat_conf




In [2]:
MIN_MAX = {
    "Weeks": (-5.0, 133.0),
    "FVC": (827.0, 6399.0),
    "Percent": (28.877577, 153.145378),
    "Age": (49.0, 88.0),
    "typical_fvc": (827.0, 6399.0),
}


def create_typical_fvc(df):
    df = df.copy()
    df["typical_fvc"] = df["FVC"] / df["Percent"].replace(0, np.nan) * 100.0
    df["typical_fvc"] = df["typical_fvc"].fillna(df["FVC"])
    return df


def normalize(df):
    df = df.copy()
    for feature, (fmin, fmax) in MIN_MAX.items():
        df[feature] = (df[feature] - fmin) / (fmax - fmin)
    return df


TEST_DF = create_typical_fvc(TEST_DF)
TEST_DF = normalize(TEST_DF)




In [3]:
pred_array = np.zeros((len(SAMPLE_SUBMISSION), 3), dtype="float32")

for idx, pw in enumerate(SAMPLE_SUBMISSION["Patient_Week"].values):
    patient, week_str = pw.split("_")
    week = float(week_str)

    # Use learned intercept and slope when available; otherwise fall back to baseline typical_fvc
    if patient in patient_trend:
        intercept, slope = patient_trend[patient]
        fvc_pred = intercept + slope * week
    else:
        baseline_rows = TEST_DF[TEST_DF["Patient"] == patient]
        if not baseline_rows.empty:
            baseline_row = baseline_rows.iloc[0]
            fvc_pred = baseline_row["typical_fvc"]
        else:
            fvc_pred = TRAIN_DF["FVC"].mean()

    # Clip to realistic FVC range
    fvc_pred = np.clip(fvc_pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])
    fvc_pred = max(fvc_pred, 0.0)

    # Apply patient‑specific (or global) bias correction
    bias_to_add = patient_bias.get(patient, BIAS)
    fvc_pred += bias_to_add
    fvc_pred = np.clip(fvc_pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])

    # Confidence (σ) – patient‑specific if computed, else global constant
    conf = patient_confidence.get(patient, CONSTANT_CONFIDENCE)

    pred_array[idx, 0] = 0.0  # placeholder (unused)
    pred_array[idx, 1] = fvc_pred
    pred_array[idx, 2] = conf




In [4]:
SAMPLE_SUBMISSION["FVC"] = pred_array[:, 1].astype("int")
SAMPLE_SUBMISSION["Confidence"] = pred_array[:, 2].astype("int")
SAMPLE_SUBMISSION.to_csv("submission.csv", index=False)
print("Submission written to submission.csv")

Submission written to submission.csv
